### Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotnine import ggplot, aes, geom_point, scale_x_log10, scale_y_continuous, scale_color_manual, labs, theme_bw, theme, element_text
import matplotlib.gridspec as gridspec
import os
import pickle

group = "Panthera_leo"
group = "Ceratotherium_simum"
group = "Rhinoceros_unicornis"
#group = "Boselaphus_tragocamelus"
ref = "Tragelaphus_eurycerus_isaaci"


if os.getcwd().startswith("/home/lakrids"):
    path_prefix = "/home/lakrids/GenomeDK"
else:
    path_prefix = "/faststorage/project/"
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/population_list.txt") as f:
        pops = [line.strip() for line in f]
population_name = pops[0]
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
    parameters = pickle.load(file)

if group == "Boselaphus_tragocamelus":
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref}/ref/regions_{ref}_updated.txt", delimiter="\t")
else: 
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/ref/regions_{group}_updated.txt", delimiter="\t")

    
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}_filtered.txt") as f:
    samples = [line.strip() for line in f if line.strip()]


### File Input & First Data Impression

In [2]:
df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

chromosomes = df_regions[
        (df_regions["female_ploidy"] == 2) &
        (df_regions["male_ploidy"] == 2) &
        (df_regions["end"] > 20000000)
    ]["region"].tolist()

# Build a contig length dict from the regions df
contig_lengths = df_regions.set_index("region")["end"].to_dict()

df_stats = df_stats[["sample"] + chromosomes]
# Divide each chromosome column by its contig lengths
for chrom in df_stats.columns:
    if chrom in contig_lengths:
        df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
df_stats

,sample,CM053518.1,CM053519.1,CM053520.1,CM053521.1,CM053522.1,CM053523.1,CM053524.1,CM053525.1,CM053526.1,...,CM053549.1,CM053550.1,CM053551.1,CM053552.1,CM053553.1,CM053554.1,CM053555.1,CM053556.1,CM053557.1,CM053558.1
0,SAMD00283119,0.012914,0.009118,0.010327,0.009371,0.010758,0.013480,0.010063,0.009729,0.010099,...,0.009784,0.009190,0.012401,0.009555,0.010037,0.010076,0.011976,0.009936,0.009943,0.023293
1,SAMN14122078,0.125180,0.116926,0.120906,0.114736,0.107948,0.108019,0.115975,0.118020,0.114853,...,0.113266,0.106097,0.121781,0.107006,0.125069,0.112919,0.117537,0.121185,0.114875,0.195984
2,SAMN17775827,0.022912,0.013899,0.013638,0.013938,0.016845,0.019875,0.016187,0.013371,0.013619,...,0.017530,0.017721,0.034524,0.024547,0.016002,0.014699,0.015085,0.015277,0.015129,0.026565
3,mappability,0.011493,0.007980,0.008996,0.008109,0.008328,0.008749,0.008564,0.007789,0.008195,...,0.007859,0.008504,0.010146,0.009317,0.009112,0.008317,0.009942,0.007752,0.008707,0.010178
4,coverage,0.140455,0.124443,0.127177,0.121773,0.119229,0.120255,0.124984,0.124966,0.121807,...,0.123895,0.116879,0.145397,0.124251,0.133939,0.121124,0.125101,0.129336,0.123200,0.207826
5,final_merged,0.449368,0.380980,0.390474,0.383543,0.378679,0.381256,0.385316,0.374591,0.379955,...,0.395677,0.379154,0.509224,0.443157,0.418342,0.396779,0.416901,0.389800,0.373418,0.680366


In [ ]:
df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

In [7]:
genus_list      = ["Loxodonta", "Elephas", "Boselaphus", "Panthera", "Rhinoceros", "Ceratotherium", "Diceros"]
#genus_list = ["Elephas"]
population_name = "pop"

data            = pd.concat([pd.read_table(f) for f in [f"{path_prefix}/megaFauna/sa_megafauna/metadata/samples_{genus}.txt" for genus in genus_list]], ignore_index=True) # add all SRR accession from genus_list in a single data frame
data            = data.reset_index(drop=True)
ref_folders     = sorted(set(data.REFERENCE_FOLDER)) # list of references needed to map the SRR accessions

references      = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/metadata/references.txt")
references      = references.loc[references.REFERENCE_FOLDER.isin(ref_folders)] # filter references to keep only the ones needed to map the SRR accessions
references      = references.reset_index(drop=True)

## make data frame that contains names of species-specific folders and the reference folders used to map the species
species_and_refs = pd.DataFrame({"FOLDER": data.FOLDER, "REFERENCE_FOLDER": data.REFERENCE_FOLDER, "GVCF_FOLDER": [data.GENUS.iloc[jj] + "_" + data.SPECIES.iloc[jj] for jj in range(data.shape[0])]}).drop_duplicates()
species_and_refs = species_and_refs.reset_index(drop=True)

## merge dataframes to have species-specific folder, reference folder and fastq ftps in same dataframe
species_and_refs = species_and_refs.merge(references, how = "left")

##########################################
#    	     --- Preparations ---
##########################################

for i in range(species_and_refs.shape[0]):
    # Initialising folders and variables for putting in the functions
    group      = species_and_refs.FOLDER[i]
    ref_folder = species_and_refs.REFERENCE_FOLDER[i]
    
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
        parameters = pickle.load(file)    

    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref_folder}/ref/regions_{ref_folder}_updated.txt", delimiter="\t")
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}.txt") as f:
        samples = [line.strip() for line in f if line.strip()]

    df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

    chromosomes = df_regions[
            df_regions["region"].str.startswith("NC") &
            (df_regions["female_ploidy"] == 2) &
            (df_regions["male_ploidy"] == 2)
        ]["region"].tolist()

    # Build a contig length dict from the regions df
    contig_lengths = df_regions.set_index("region")["end"].to_dict()

    df_stats = df_stats[["sample"] + chromosomes]
    # Divide each chromosome column by its contig lengths
    for chrom in df_stats.columns:
        if chrom in contig_lengths:
            df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
    df_stats

    df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/mask_stats_{group}.txt")

KeyError: "None of [Index(['sample', 'NC_087342.1', 'NC_087343.1', 'NC_087344.1', 'NC_087345.1',\n       'NC_087346.1', 'NC_087347.1', 'NC_087348.1', 'NC_087349.1',\n       'NC_087350.1', 'NC_087351.1', 'NC_087352.1', 'NC_087353.1',\n       'NC_087354.1', 'NC_087355.1', 'NC_087356.1', 'NC_087357.1',\n       'NC_087358.1', 'NC_087359.1', 'NC_087360.1', 'NC_087361.1',\n       'NC_087362.1', 'NC_087363.1', 'NC_087364.1', 'NC_087365.1',\n       'NC_087366.1', 'NC_087367.1', 'NC_087368.1'],\n      dtype='object')] are in the [columns]"